In [132]:
import numpy as np
import pandas as pd
X_train_df = pd.read_csv('X_train.csv', skiprows=1, header=None)
y_train_df = pd.read_csv('y_train.csv', skiprows=1, header=None)
X_test_df = pd.read_csv('X_test.csv', skiprows=1, header=None)
# drop the id column
X_train = X_train_df.values[:, 1:]
y_train = y_train_df.values[:, 1:]
X_test = X_test_df.values[:, 1:]
print(X_train.shape, y_train.shape, X_test.shape)


(1212, 832) (1212, 1) (776, 832)


In [133]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

# Randomly split the data into training and validation sets (80/20)
X_train, X_val, y_train, y_val = train_test_split(
X_train, y_train, test_size=0.2, random_state=13)
# Impute missing values with the mean of each column (training statistics only)

scaler = RobustScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)
X_val = scaler.transform(X_val)
imputer = SimpleImputer(missing_values=np.nan, strategy="mean")
imputer.fit(X_train)
X_train_t = imputer.transform(X_train)

In [134]:
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.impute import KNNImputer
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import RidgeCV
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestRegressor

# 1. Feature selection lineare calcolata sui dati imputati con la media
#selection = SelectKBest(score_func=f_regression, k=150)
selection = RFE(estimator = Ridge(), n_features_to_select=100)
selection.fit(X_train_t, y_train.ravel())
mask = selection.get_support()

# 2. Taglio a 100 colonne PRIMA del KNNImputer (mantenendo i NaN intatti)
X_train_sub = X_train[:, mask]
X_val_sub   = X_val[:, mask]
X_test_sub  = X_test[:, mask]

# 3. KNNImputer lavora ORA su sole 100 feature pulite e già scalate
multi_imputer = KNNImputer(missing_values=np.nan, n_neighbors=20, weights="distance")
X_train_clean = multi_imputer.fit_transform(X_train_sub)
X_val_clean   = multi_imputer.transform(X_val_sub)
X_test_clean  = multi_imputer.transform(X_test_sub)

# 4. Rimozione Outlier SOLO sul Train Set
iso = IsolationForest(contamination=0.025, random_state=42)
inliers = (iso.fit_predict(X_train_clean) == 1)

X_train_final = X_train_clean[inliers]
y_train_final = y_train[inliers]

# 5. RidgeCV per scegliere in automatico l'alpha ottimale
#alphas = np.logspace(0, 4, 40)  # Testa da 1 a 10.000
#regressor = RidgeCV(alphas=alphas, cv=5, scoring='r2')
regressor = RandomForestRegressor(
    n_estimators=400,        
    max_depth=10,            
    min_samples_leaf=4,      
    max_features='sqrt',     
    random_state=42,
    n_jobs=-1                
)
from sklearn.svm import SVR

regressor = SVR(kernel='rbf', C=10.0, epsilon=0.1)
regressor.fit(X_train_final, y_train_final.ravel())

# 6. Valutazione
train_score = r2_score(y_train_final, regressor.predict(X_train_final))
val_score   = r2_score(y_val, regressor.predict(X_val_clean))

#print(f"Alpha ottimale scelto: {regressor.alpha_:.2f}")
print(f"R2 Train: {train_score:.4f}")
print(f"R2 Val:   {val_score:.4f}")
#print("Peso massimo (in modulo):", np.max(np.abs(regressor.coef_)))

predictions = regressor.predict(X_test_clean)
test_ids = X_test_df.values[:, 0]

# 3. Crea il DataFrame con l'esatta struttura richiesta da Kaggle
submission = pd.DataFrame({
    'id': test_ids.astype(int),  # Assicurati che l'ID sia intero e non float
    'age': predictions.ravel()   # Sostituisci 'age' col nome target esatto della competizione
})

# 4. Salva il file CSV senza l'indice numerico di riga di pandas
submission.to_csv('submission.csv', index=False)

R2 Train: 0.8636
R2 Val:   0.4506


In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_regression
# Select the 100 features with the highest mutual information with the age
selection = SelectKBest(mutual_info_regression, k=100).fit(X_train_t, y_train.ravel())

multi_imputer = KNNImputer(missing_values=np.nan, n_neighbors=10)
multi_imputer.fit(X_train)
X_train = multi_imputer.transform(X_train)
X_val = multi_imputer.transform(X_val)
X_test = multi_imputer.transform(X_test)

X_train = selection.transform(X_train)
X_test = selection.transform(X_test)
X_val = selection.transform(X_val)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
# Train a linear regression model
regressor = Ridge(alpha=0.7)
regressor.fit(X_train, y_train)
# Evaluate the model on the training and validation sets
train_score = r2_score(y_train, regressor.predict(X_train))
val_score = r2_score(y_val, regressor.predict(X_val))
print(train_score, val_score)
print("Peso massimo (in modulo):", np.max(np.abs(regressor.coef_)))
print("Media dei pesi (in modulo):", np.mean(np.abs(regressor.coef_)))
